# Reading the Street Before You Walk It

**Purpose:** a short, two-metric introduction to Space Syntax for the field trip — not the
full technical analysis. **Full technical version, all metrics:**
[`03-NA01-SpaceSyntax.ipynb`](03-NA01-SpaceSyntax.ipynb) · **Concept glossary:**
[`03-NA01-SpaceSyntax_Overview.ipynb`](../documentations/03-NA01-SpaceSyntax_Overview.ipynb)

Urban morphology — the physical shape of a city — is usually read from a plan: block sizes,
building footprints, street widths. Space Syntax reads something else from the same street
network's geometry: **how a street behaves**, purely from how it connects to everything around
it, before a single person has ever walked it. No traffic counts, no survey, no land-use data —
just the shape of the network.

That prediction is testable. Two numbers, computed below, describe something you can actually
*feel* standing on a street:

- **Integration** — how easy this street is to find and get back to from anywhere else in the
  area. High-Integration streets are legible: you don't get lost on them.
- **Choice** (at a pedestrian radius, R800 ≈ 800 m) — how much of everyone *else's* shortest
  walking route happens to pass through this exact street, whether they're headed there or not.
  High-Choice streets carry footfall that has nothing to do with what's actually on them.

A street that scores high on both is the classic candidate for a lively main street —
shopfronts, people lingering, easy to navigate. A street low on both is a quiet residential lane
you could easily miss. This notebook computes both for the streets around the case study area,
so you have a prediction to test against what you actually notice on the ground.

## 1. Setup

In [1]:
# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd
import folium

from sitex.core.config import CityConfig
from sitex.network import spacesyntax as ss

city = CityConfig(
    place_name="Phường Pleiku, Gia Lai, Vietnam",
    local_lat=13.9833,
    local_lon=108.0000,
    slug="phường_pleiku",
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
)

Running in: Local environment


## 2. Load the network you'll actually walk

The field trip happens on foot, so this uses the **pedestrian (walk) network**, not the drivable
street network the technical notebook analyses. A road that's easy to drive but has no footway,
or that a pedestrian would never realistically follow, shouldn't count as part of what you'll
experience walking it.

In [2]:
STREETS_FILE = city.data_dir / "osm" / f"{city.slug}_walk.gpkg"
streets = gpd.read_file(STREETS_FILE, layer="edges").to_crs(epsg=city.local_epsg)
print(f"Loaded {len(streets):,} walkable street edges  (EPSG:{city.local_epsg})")

Loaded 3,344 walkable street edges  (EPSG:32649)


## 3. Build the axial map

Space Syntax doesn't reason over every individual OSM edge — it merges connected street segments
that continue in roughly the same direction (within 20°) into single long lines, called **axial
lines**. This is a computational stand-in for how you actually perceive a street while walking
it: a street that bends slightly at a junction still reads as "the same street," not as two.

In [3]:
axial_map = ss.graph_to_axial_map(streets, angular_tolerance=20.0)
axial_G = ss.gdf_to_nx_graph(axial_map, id_col="axial_id")
print(
    f"{len(streets):,} walked edges merged into {len(axial_map):,} axial lines "
    f"({axial_G.number_of_nodes():,} nodes, {axial_G.number_of_edges():,} edges)"
)

3,344 walked edges merged into 1,615 axial lines (2,494 nodes, 1,604 edges)


## 4. Two questions, two numbers

Both metrics come from the same axial graph — no separate segment or angular-turn analysis,
which the full technical notebook adds for a finer-grained but harder-to-read picture.
`radii={"R800": 800}` restricts the Choice calculation to exactly the one radius this notebook
uses, rather than computing all five (R400…Rn) as the technical version does.

In [4]:
integration = ss.axial_integration(axial_G)
choice_r800 = ss.axial_choice_multiscale(axial_G, radii={"R800": 800})["R800"]

field_gdf = ss.metrics_to_gdf(
    axial_G,
    {"integration": integration, "choice_R800": choice_r800},
    crs=axial_map.crs,
    id_col="axial_id",
)
field_gdf[["integration", "choice_R800"]].describe()

,integration,choice_R800
count,1604.000000,1604.000000
mean,3.127325,9.628429
std,6.874262,18.067499
min,0.000000,0.000000
25%,0.000000,2.000000
50%,0.000000,2.000000
75%,3.750000,10.000000
max,47.215736,210.000000


## 5. Compare the two maps

One interactive map, two toggleable layers — switch between them with the layer control in the
top-right corner. Hover any street for its exact values. (This project's environment has a known
crash in `GeoDataFrame.plot()` via matplotlib — see the note in
[`03-NA01-SpaceSyntax.ipynb`](03-NA01-SpaceSyntax.ipynb) §8 — so, like the rest of the Space
Syntax notebooks here, this renders with folium instead.)

In [5]:
field_wgs = field_gdf.to_crs(epsg=4326)
tips = ["integration", "choice_R800"]

m = folium.Map(location=(city.local_lat, city.local_lon), zoom_start=15,
                tiles="cartodbpositron", control_scale=True)
ss.add_metric_layer(m, field_wgs, "integration", "Integration — how easy this street is to FIND",
                     "RdYlBu_r", tips, id_col="axial_id", show=True)
ss.add_metric_layer(m, field_wgs, "choice_R800", "Choice R800 — how much foot traffic PASSES THROUGH",
                     "YlOrRd", tips, id_col="axial_id", show=False)
folium.LayerControl(collapsed=False).add_to(m)

map_path = city.output_dir / "fieldtrip_spacesyntax_preview.html"
m.save(str(map_path))
print(f"Saved {map_path} — open on a phone/tablet on the trip, or screenshot each layer for a printout")
m

Saved ..\outputs\fieldtrip_spacesyntax_preview.html — open on a phone/tablet on the trip, or screenshot each layer for a printout


## 6. Field-trip assignment

Before you go, pick 2–3 street segments from each combination below, using the maps above:

| Combination | What the model predicts | What to check on the ground |
|---|---|---|
| High Integration + High Choice | Legible *and* busy — likely the main street | Shopfronts? People lingering, not just passing through? |
| High Integration + Low Choice | Easy to find, but off the through-routes | Quiet despite being central — why? |
| Low Integration + High Choice | Hard to find your way back to, but everyone passes through | A bypass/through-route — does it feel like "a place" at all? |
| Low Integration + Low Choice | Neither — a genuine backstreet | Could you find this again without a map? |

Record what you actually notice — pedestrian density, shopfronts, how easily you could
re-orient yourself — against what the map predicted. **Where the model is wrong is often more
interesting than where it's right**: it usually means something about that street isn't captured
by network geometry alone (a market, a school gate, a landmark, a wall that blocks a sightline the
map doesn't know about).

## Where to go next

| To do this analysis | Open |
|---|---|
| Full metric set — Connectivity, Choice at 5 radii, Angular Choice, NACH, Reach, scenario testing | [`03-NA01-SpaceSyntax.ipynb`](03-NA01-SpaceSyntax.ipynb) |
| What each metric means and why, in more depth | [`03-NA01-SpaceSyntax_Overview.ipynb`](../documentations/03-NA01-SpaceSyntax_Overview.ipynb) |
| Pair Choice/Integration with what's actually on these streets | [`03-NA02-POI_Accessibility.ipynb`](03-NA02-POI_Accessibility.ipynb) |
| Compare against observed pedestrian activity (Gehl-style heatmaps) | [`04-VIZ-Gehl_Webmap.ipynb`](04-VIZ-Gehl_Webmap.ipynb) |